In [11]:
import wandb

In [12]:
api = wandb.Api()
project = "lupos/flexible-printed-network"

In [13]:
for run in api.runs(project):
    # List all artifacts for this run
    artifacts = list(run.logged_artifacts())
    found_best = False
    for artifact in artifacts:
        # Check if "best" is one of the aliases
        if "best" in artifact.aliases:
            artifact.download(root=f"./checkpoints/{run.name}")
            print(f"Downloaded best model {artifact.name} for run {run.name}")
            found_best = True
    if not found_best:
        print(f"No best model found for run {run.name}")

wandb:   1 of 1 files downloaded.  


Downloaded best model model-tbfwpi3o:v0 for run PSNN_wSurrGPT_wFaults_acuteinflammation_run1


wandb:   1 of 1 files downloaded.  


Downloaded best model model-101ezyhl:v0 for run PSNN_wSurrGPT_woFaults_acuteinflammation_run1


wandb:   1 of 1 files downloaded.  


...

wandb:   1 of 1 files downloaded.  


Downloaded best model model-q3esyzc5:v0 for run PSNN_wSurrGPT_wFaults_cbf_run2


wandb:   1 of 1 files downloaded.  


Downloaded best model model-7n8se45t:v0 for run PSNN_wSurrGPT_woFaults_acuteinflammation_run3


KeyboardInterrupt: 

In [3]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
import sys
import os
import snntorch as snn 
import tempfile

from utils.PrintedSpikingNN_lP_New import LightningPrintedSpikingNetwork
from surrogate.utils.MyTransformer_lP import GPTLightning, GPT

from utils.Loader import GetDataLoader
from utils.configuration import load_args
from argparse import Namespace

from pytorch_lightning import Trainer
import numpy as np


# Fix for PyTorch 2.6+ security settings
torch.serialization.add_safe_globals([Namespace])


def get_power_metrics(
    ckpt_path: str,
    surr_ckpt_path: str,
    test_loader,
    num_mc_draws_per_batch: int = 10,
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
):
    print(f"--- Preparing model config and patching surrogate ---")
    model_config = GPT.get_default_config()
    model_config.model_type = 'gpt-nano'
    model_config.n_extra_params = 4
    model_config.block_size = 104

    surr_ckpt = torch.load(surr_ckpt_path, map_location='cpu', weights_only=False)
    if 'hyper_parameters' not in surr_ckpt:
        surr_ckpt['hyper_parameters'] = {}
    surr_ckpt['hyper_parameters']['model_config'] = model_config
    surr_ckpt['hyper_parameters']['max_epochs'] = 100

    with tempfile.NamedTemporaryFile(suffix=".ckpt", delete=False) as tmp:
        torch.save(surr_ckpt, tmp.name)
        patched_surr_path = tmp.name

    try:
        print(f"--- Loading Full Network from {ckpt_path} ---")
        model = LightningPrintedSpikingNetwork.load_from_checkpoint(
            ckpt_path,
            map_location=device,
            weights_only=False,
            model_class=GPTLightning,
            ckpt_path=patched_surr_path,
            train_loader=None,
            valid_loader=None,
            test_loader=test_loader,
            surrogate_gradient=snn.surrogate.atan(),
            train_dataset=None,
            valid_dataset=None,
            strict=False
        )
        model.to(device)
        model.eval()

        # We will compute the powers manually to avoid running the full on_test_epoch_end (which has a bug in your setup)
        def compute_power_for_mode(mode: str, draws: int):
            model.args.fault_mode = mode
            model.args.use_interpolation = False
            if hasattr(model.network, "UpdateArgs"):
                model.network.UpdateArgs(model.args)

            powers_fault = []
            powers_nofault = []

            print(f"--- Computing power for mode='{mode}' with {draws} MC draws per batch ---")
            with torch.no_grad():
                for xb, yb in tqdm(test_loader, desc=f"Mode {mode}"):
                    xb = xb.to(device)
                    yb = yb.to(device)

                    for _ in range(draws):
                        # Run forward (power and last_fault_info are set inside the network)
                        _ = model.network(xb)

                        power_val = float(model.network.power.detach().cpu().item())
                        fault_info = getattr(model.network, "last_fault_info", None)

                        if fault_info is not None:
                            powers_fault.append(power_val)
                        else:
                            powers_nofault.append(power_val)

            mean_fault = float(np.mean(powers_fault)) if powers_fault else float("nan")
            mean_nofault = float(np.mean(powers_nofault)) if powers_nofault else float("nan")

            return mean_fault, mean_nofault

        # Compute no-fault power (mode "none")
        # Use the same number of draws for consistency (though deterministic)
        _, none_nofault_power = compute_power_for_mode("none", num_mc_draws_per_batch)

        # Compute single-fault power (mode "single")
        single_fault_power, _ = compute_power_for_mode("single", num_mc_draws_per_batch)

        # If for some reason single mode produced no-fault draws, you could fallback, but normally all draws have a fault
        if np.isnan(single_fault_power):
            print("Warning: No fault draws in single mode – this should not happen.")

        print(f"test_power_mode_single_fault = {single_fault_power}")
        print(f"test_power_mode_none_nofault = {none_nofault_power}")

        return pd.DataFrame([{
            "test_power_mode_single_fault": single_fault_power,
            "test_power_mode_none_nofault": none_nofault_power
        }])

    finally:
        if os.path.exists(patched_surr_path):
            os.remove(patched_surr_path)

In [6]:
import os
import pandas as pd
import traceback
from utils.Loader import GetDataLoader
from utils.configuration import load_args

# ========== Configuration ==========
CHECKPOINTS_ROOT = "./checkpoints"
SURR_CKPT_PATH = "surrogate/models/BaselineGPT/GPT_Nano_run1-gpt-nano-epoch=185-val_loss=0.36.ckpt"
OUT_CSV = "all_runs_power_usage.csv"
BATCH_SIZE = 64  # small; adjust if you want bigger
DEVICE = "cpu"  # safer default; change to "cuda" if you want GPU and available

# ========== Dataset lists (copied from your GetDataLoader) ==========
normal_datasets = ['Dataset_acuteinflammation.ds',  # 0
                   'Dataset_balancescale.ds',  # 1
                   'Dataset_breastcancerwisc.ds',  # 2
                   'Dataset_cardiotocography3clases.ds',  # 3
                   'Dataset_energyy1.ds',  # 4
                   'Dataset_energyy2.ds',  # 5
                   'Dataset_iris.ds',  # 6
                   'Dataset_mammographic.ds',  # 7
                   'Dataset_pendigits.ds',  # 8
                   'Dataset_seeds.ds',  # 9
                   'Dataset_tictactoe.ds',  # 10
                   'Dataset_vertebralcolumn2clases.ds',  # 11
                   'Dataset_vertebralcolumn3clases.ds']  # 12

temporized_datasets = normal_datasets[:]  # as in your code

temporal_datasets = ['Dataset_cbf.tsds', # 0
                     'Dataset_distalphalanxtw.tsds', # 1 
                     'Dataset_freezerregulartrain.tsds', # 2 
                     'Dataset_freezersmalltrain.tsds', # 3
                     'Dataset_gunpointagespan.tsds', # 4
                     'Dataset_gunpointmaleversusfemale.tsds', # 5
                     'Dataset_gunpointoldversusyoung.tsds', # 6
                     'Dataset_middlephalanxoutlineagegroup.tsds', # 7
                     'Dataset_mixedshapesregulartrain.tsds', # 8
                     'Dataset_powercons.tsds', # 9
                     'Dataset_proximalphalanxoutlinecorrect.tsds', # 10
                     'Dataset_selfregulationscp2.tsds', # 11
                     'Dataset_slope.tsds', # 12
                     'Dataset_smoothsubspace.tsds', # 13
                     'Dataset_symbols.tsds'] # 14


# ========== Build name -> (task, index) mapping ==========
def shortname_from_dataset_entry(entry):
    # remove "Dataset_" prefix and extension (.ds / .tsds)
    name = entry
    if name.startswith("Dataset_"):
        name = name[len("Dataset_"):]
    for ext in (".ds", ".tsds"):
        if name.endswith(ext):
            name = name[:-len(ext)]
    return name.lower()

dataset_map = {}  # shortname -> (task, index)
dataset_map = {}
for i, e in enumerate(normal_datasets):
    dataset_map[shortname_from_dataset_entry(e)] = ("temporized", i)

for i, e in enumerate(temporized_datasets):
    dataset_map[shortname_from_dataset_entry(e)] = ("temporized", i)

for i, e in enumerate(temporal_datasets):
    dataset_map[shortname_from_dataset_entry(e)] = ("temporal", i)

# Optional: print mapping keys for debugging
# print("Known dataset keys:", sorted(dataset_map.keys()))

# ========== Main loop ==========
all_rows = []

# helper to find dataset key in run_name
def find_dataset_for_run(run_name):
    rn = run_name.lower()
    # try to find the longest matching key (to avoid accidental short matches)
    matches = [k for k in dataset_map.keys() if k in rn]
    if not matches:
        return None
    # pick the longest match (e.g., 'breastcancerwisc' vs 'breastcancer')
    best = max(matches, key=len)
    return best

# Iterate runs
for run_name in sorted(os.listdir(CHECKPOINTS_ROOT)):
    run_path = os.path.join(CHECKPOINTS_ROOT, run_name)
    if not os.path.isdir(run_path):
        continue

    print(f"\n=== Processing run folder: {run_name} ===")
    # find checkpoint file
    ckpt_files = [f for f in os.listdir(run_path) if f.endswith(".ckpt")]
    if not ckpt_files:
        print(f"  -> No .ckpt files found in {run_path}, skipping.")
        continue
    ckpt_path = os.path.join(run_path, ckpt_files[0])  # pick first .ckpt (adjust if needed)
    print(f"  -> using checkpoint: {ckpt_path}")

    # find dataset
    dataset_key = find_dataset_for_run(run_name)
    if dataset_key is None:
        print(f"  -> WARNING: could not infer dataset from run name '{run_name}'. Skipping.")
        continue

    task, dataset_idx = dataset_map[dataset_key]
    if task not in ("temporal", "temporized"):
        print(f"  -> skipping dataset {dataset_key} (task={task})")
        continue

    # Build args overrides and get test_loader appropriately
    try:
        overrides = {
            "DATASET": dataset_idx,
            "task": task,
            "DEVICE": DEVICE,
            # keep other defaults as necessary; small batch is safe
        }
        # ensure batch size small and defined
        # load_args and GetDataLoader are assumed to be available in your environment
        args = load_args(overrides=overrides)

        # For 'split' task GetDataLoader returns lists; we need to pick the right index
        if task == "split":
            test_loaders, infos = GetDataLoader(args, 'test', batch_size=BATCH_SIZE)
            # split_manufacture ordering matches the index we have in dataset_idx
            try:
                test_loader = test_loaders[dataset_idx]
            except Exception as e:
                print(f"  -> ERROR selecting split test loader: {e}\n  -> skipping run.")
                continue
        else:
            test_loader, info = GetDataLoader(args, 'test', batch_size=BATCH_SIZE)
    except Exception as e:
        print(f"  -> ERROR creating data loader for run {run_name}: {e}")
        traceback.print_exc()
        continue

    # Run get_power_metrics (assumes get_power_metrics is defined and in scope)
    try:
        print("  -> computing power metrics (this may take time)...")
        df_metrics = get_power_metrics(ckpt_path, SURR_CKPT_PATH, test_loader, 10)
        # expected one-row df with columns test_power_mode_single_fault and test_power_mode_none_nofault
        single_val = float(df_metrics["test_power_mode_single_fault"].iloc[0])
        none_val = float(df_metrics["test_power_mode_none_nofault"].iloc[0])

        all_rows.append({
            "run_name": run_name,
            "fault_mode": "single_fault",
            "power_usage": single_val
        })
        all_rows.append({
            "run_name": run_name,
            "fault_mode": "no_fault",
            "power_usage": none_val
        })

        print(f"  -> done. single_fault={single_val:.4f}, no_fault={none_val:.4f}")

    except Exception as e:
        print(f"  -> ERROR running get_power_metrics for {run_name}: {e}")
        traceback.print_exc()
        # continue to next run

# ========== Finalize ==========
final_df = pd.DataFrame(all_rows)
print("\n=== Summary ===")
print(final_df.head(20))
final_df.to_csv(OUT_CSV, index=False)
print(f"\nSaved {OUT_CSV} ({len(final_df)} rows, {len(final_df)//2} runs approx).")



=== Processing run folder: PSNN_wSurrGPT_wFaults_acuteinflammation_run1 ===
  -> using checkpoint: ./checkpoints/PSNN_wSurrGPT_wFaults_acuteinflammation_run1/model.ckpt
  -> computing power metrics (this may take time)...
--- Preparing model config and patching surrogate ---
--- Loading Full Network from ./checkpoints/PSNN_wSurrGPT_wFaults_acuteinflammation_run1/model.ckpt ---
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k
number of par

Mode none: 100%|███████| 1/1 [00:06<00:00,  6.09s/it]


--- Computing power for mode='single' with 10 MC draws per batch ---


Mode single: 100%|█████| 1/1 [00:06<00:00,  6.34s/it]


test_power_mode_single_fault = 1.7401182594767307e-05
test_power_mode_none_nofault = 1.559290467412211e-05
  -> done. single_fault=0.0000, no_fault=0.0000

=== Processing run folder: PSNN_wSurrGPT_wFaults_acuteinflammation_run2 ===
  -> using checkpoint: ./checkpoints/PSNN_wSurrGPT_wFaults_acuteinflammation_run2/model.ckpt
  -> computing power metrics (this may take time)...
--- Preparing model config and patching surrogate ---
--- Loading Full Network from ./checkpoints/PSNN_wSurrGPT_wFaults_acuteinflammation_run2/model.ckpt ---
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k
number of parameters: 90k
number of parameters: 3k
number of

...

test_power_mode_single_fault = 0.0012844099779613315
test_power_mode_none_nofault = 0.001140997395850718
  -> done. single_fault=0.0013, no_fault=0.0011

=== Processing run folder: PSNN_wSurrGPT_woFaults_cbf_run2 ===
  -> using checkpoint: ./checkpoints/PSNN_wSurrGPT_woFaults_cbf_run2/model.ckpt
  -> computing power metrics (this may take time)...
--- Preparing model config and patching surrogate ---
--- Loading Full Network from ./checkpoints/PSNN_wSurrGPT_woFaults_cbf_run2/model.ckpt ---
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k
number of parameters: 90k
number of parameters: 3k
number of parameters: 3k
number of parameters: 3k


Mode none: 100%|███████| 2/2 [00:08<00:00,  4.22s/it]


--- Computing power for mode='single' with 10 MC draws per batch ---


Mode single: 100%|█████| 2/2 [00:09<00:00,  4.50s/it]

test_power_mode_single_fault = 0.0019898045749869196
test_power_mode_none_nofault = 0.0016234145732596517
  -> done. single_fault=0.0020, no_fault=0.0016

=== Summary ===
                                         run_name    fault_mode  power_usage
0    PSNN_wSurrGPT_wFaults_acuteinflammation_run1  single_fault     0.000017
1    PSNN_wSurrGPT_wFaults_acuteinflammation_run1      no_fault     0.000016
2    PSNN_wSurrGPT_wFaults_acuteinflammation_run2  single_fault     0.000146
3    PSNN_wSurrGPT_wFaults_acuteinflammation_run2      no_fault     0.000142
4                  PSNN_wSurrGPT_wFaults_cbf_run1  single_fault     0.002294
5                  PSNN_wSurrGPT_wFaults_cbf_run1      no_fault     0.001831
6                  PSNN_wSurrGPT_wFaults_cbf_run2  single_fault     0.005329
7                  PSNN_wSurrGPT_wFaults_cbf_run2      no_fault     0.005014
8   PSNN_wSurrGPT_woFaults_acuteinflammation_run1  single_fault     0.000014
9   PSNN_wSurrGPT_woFaults_acuteinflammation_run1      no_f